# Complete Transformer Implementation from Scratch

**Goal**: Build a complete Transformer architecture understanding every mathematical detail.

**Based on**: *Attention Is All You Need* (Vaswani et al., 2017)

---

## Table of Contents

1. [Introduction](#introduction)
2. [Scaled Dot-Product Attention](#scaled-attention)
3. [Multi-Head Attention](#multi-head)
4. [Position-wise Feed-Forward Networks](#ffn)
5. [Positional Encoding](#positional)
6. [Encoder Layer](#encoder)
7. [Decoder Layer](#decoder)
8. [Complete Transformer](#complete)
9. [Training Example](#training)
10. [Attention Visualization](#visualization)
11. [Complexity Analysis](#complexity)
12. [Gradient Checking](#gradcheck)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Tuple
import time

# For gradient checking
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 1. Introduction <a name="introduction"></a>

### The Transformer Architecture

The Transformer revolutionized NLP by replacing recurrence with **self-attention**.

**Key Innovation**: Parallel processing of sequences (unlike RNNs/LSTMs)

### Architecture Overview

```
Input Sequence
    ↓
Embedding + Positional Encoding
    ↓
┌─────────────────┐
│  Encoder Stack  │  (N layers)
│  - Multi-Head   │
│    Attention    │
│  - Feed Forward │
└─────────────────┘
    ↓
┌─────────────────┐
│  Decoder Stack  │  (N layers)
│  - Masked Attn  │
│  - Cross Attn   │
│  - Feed Forward │
└─────────────────┘
    ↓
Output Probabilities
```

### Mathematical Foundation

The core operation is **scaled dot-product attention**:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Where:
- $Q$ (Query): What am I looking for?
- $K$ (Key): What do I contain?
- $V$ (Value): What information do I have?
- $d_k$: Dimension of keys (for scaling)

---

## 2. Scaled Dot-Product Attention <a name="scaled-attention"></a>

### Mathematical Derivation

**Step 1: Compute attention scores**
$$
\text{scores} = QK^T \in \mathbb{R}^{n \times n}
$$

**Step 2: Scale by $\sqrt{d_k}$**

Why scaling?
- Without scaling: $\text{Var}(q^T k) = d_k$ (grows with dimension)
- With scaling: $\text{Var}\left(\frac{q^T k}{\sqrt{d_k}}\right) = 1$
- Prevents softmax saturation for large $d_k$

**Step 3: Apply softmax**
$$
\text{weights} = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)
$$

**Step 4: Weighted sum of values**
$$
\text{output} = \text{weights} \cdot V
$$

### Implementation

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention.
    
    Args:
        Q: Queries, shape (batch, seq_len, d_k)
        K: Keys, shape (batch, seq_len, d_k)
        V: Values, shape (batch, seq_len, d_v)
        mask: Optional mask, shape (batch, seq_len, seq_len)
              Use -inf for positions that should not attend
    
    Returns:
        output: Attention output, shape (batch, seq_len, d_v)
        attention_weights: Attention weights, shape (batch, seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute attention scores
    # (batch, seq_len, d_k) @ (batch, d_k, seq_len) -> (batch, seq_len, seq_len)
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Step 2: Scale
    scores = scores / np.sqrt(d_k)
    
    # Step 3: Apply mask (if provided)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Step 4: Softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 5: Weighted sum of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

### Example: Simple Attention

In [ ]:
# Simple example: 3 words, 4-dimensional embeddings
batch_size = 1
seq_len = 3
d_model = 4

# Create sample Q, K, V
Q = torch.randn(batch_size, seq_len, d_model)
K = torch.randn(batch_size, seq_len, d_model)
V = torch.randn(batch_size, seq_len, d_model)

output, weights = scaled_dot_product_attention(Q, K, V)

print("Input shapes:")
print(f"Q: {Q.shape}, K: {K.shape}, V: {V.shape}")
print(f"\nOutput shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")

# Visualize attention weights
plt.figure(figsize=(6, 5))
sns.heatmap(weights[0].detach().numpy(), annot=True, fmt='.3f', 
            cmap='YlOrRd', square=True, cbar_kws={'label': 'Attention Weight'})
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Attention Weights Matrix')
plt.show()

# Verify weights sum to 1
print(f"\nRow sums (should be 1.0): {weights[0].sum(dim=-1).numpy()}")

### Masking Example

**Two types of masks**:
1. **Padding mask**: Ignore padding tokens
2. **Causal mask**: Prevent attending to future positions (decoder)

In [ ]:
def create_causal_mask(seq_len):
    """
    Create lower triangular mask.
    
    Example for seq_len=4:
    [[1, 0, 0, 0],
     [1, 1, 0, 0],
     [1, 1, 1, 0],
     [1, 1, 1, 1]]
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

# Example
causal_mask = create_causal_mask(4)
print("Causal Mask:")
print(causal_mask.numpy())

# Apply to attention
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)

output, masked_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

plt.figure(figsize=(6, 5))
sns.heatmap(masked_weights[0].detach().numpy(), annot=True, fmt='.3f',
            cmap='YlOrRd', square=True, vmin=0, vmax=1)
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Masked Attention (Causal)')
plt.show()

---

## 3. Multi-Head Attention <a name="multi-head"></a>

### Why Multiple Heads?

**Problem**: Single attention focuses on one aspect
**Solution**: Multiple attention "heads" capture different relationships

Example:
- Head 1: Subject-verb relationships
- Head 2: Adjective-noun relationships
- Head 3: Long-range dependencies

### Mathematical Formulation

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O
$$

Where each head is:
$$
\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
$$

Parameters:
- $W_i^Q \in \mathbb{R}^{d_{\text{model}} \times d_k}$
- $W_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$
- $W_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$
- $W^O \in \mathbb{R}^{hd_v \times d_{\text{model}}}$

Typical values: $d_k = d_v = d_{\text{model}} / h$

### Implementation

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention module.
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        
    def split_heads(self, x, batch_size):
        """
        Split the last dimension into (num_heads, d_k).
        Transpose to shape: (batch, num_heads, seq_len, d_k)
        """
        x = x.view(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]
        
        # Linear projections
        Q = self.W_q(Q)  # (batch, seq_len, d_model)
        K = self.W_k(K)
        V = self.W_v(V)
        
        # Split into multiple heads
        Q = self.split_heads(Q, batch_size)  # (batch, num_heads, seq_len, d_k)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)
        
        # Scaled dot-product attention for each head
        d_k = Q.shape[-1]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        
        if mask is not None:
            # Expand mask for multiple heads
            mask = mask.unsqueeze(1)  # (batch, 1, seq_len, seq_len)
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_output = torch.matmul(attention_weights, V)
        
        # Concatenate heads
        # (batch, num_heads, seq_len, d_k) -> (batch, seq_len, num_heads, d_k)
        attention_output = attention_output.transpose(1, 2).contiguous()
        # -> (batch, seq_len, d_model)
        attention_output = attention_output.view(batch_size, -1, self.d_model)
        
        # Final linear projection
        output = self.W_o(attention_output)
        
        return output, attention_weights

### Example: Multi-Head Attention

In [ ]:
# Parameters
d_model = 512
num_heads = 8
batch_size = 2
seq_len = 10

# Create module
mha = MultiHeadAttention(d_model, num_heads)

# Sample input
x = torch.randn(batch_size, seq_len, d_model)

# Self-attention (Q = K = V = x)
output, attention_weights = mha(x, x, x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"  (batch_size, num_heads, seq_len, seq_len)")

# Visualize different heads
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(num_heads):
    sns.heatmap(attention_weights[0, i].detach().numpy(), 
                ax=axes[i], cmap='YlOrRd', square=True, cbar=False)
    axes[i].set_title(f'Head {i+1}')
    axes[i].set_xlabel('Key')
    axes[i].set_ylabel('Query')

plt.suptitle('Multi-Head Attention Patterns', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("\nNotice how different heads learn different attention patterns!")

---

## 4. Position-wise Feed-Forward Networks <a name="ffn"></a>

### Architecture

Applied to each position **separately and identically**:

$$
\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2
$$

Or more generally:
$$
\text{FFN}(x) = \text{activation}(xW_1 + b_1)W_2 + b_2
$$

**Typical dimensions**:
- $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$ where $d_{\text{ff}} = 4 \times d_{\text{model}}$
- $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$

**Purpose**: Add non-linearity and transform representations

### Implementation

In [ ]:
class PositionWiseFeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionWiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            output: (batch, seq_len, d_model)
        """
        # First linear + ReLU
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        # Second linear
        x = self.linear2(x)
        
        return x

### Example

In [ ]:
d_model = 512
d_ff = 2048  # 4x expansion

ffn = PositionWiseFeedForward(d_model, d_ff)

x = torch.randn(2, 10, d_model)
output = ffn(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nParameters: {sum(p.numel() for p in ffn.parameters()):,}")

---

## 5. Positional Encoding <a name="positional"></a>

### Why Needed?

**Problem**: Attention is permutation-invariant
- "The cat sat on the mat" ≡ "mat the on sat cat The"

**Solution**: Add position information to embeddings

### Sinusoidal Positional Encoding

$$
PE_{(\text{pos}, 2i)} = \sin\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right)
$$

$$
PE_{(\text{pos}, 2i+1)} = \cos\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right)
$$

Where:
- $\text{pos}$: Position in sequence (0, 1, 2, ...)
- $i$: Dimension index (0, 1, 2, ..., $d_{\text{model}}/2$)

### Properties

1. **Unique encoding** for each position
2. **Deterministic** (not learned)
3. **Relative position** can be computed:
   - $PE_{\text{pos}+k}$ is a linear function of $PE_{\text{pos}}$
4. **Extrapolation** to longer sequences

### Implementation

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding.
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        
        # Position indices: (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Division term: 10000^(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             (-np.log(10000.0) / d_model))
        
        # Apply sin to even indices
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cos to odd indices
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension: (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        
        # Register as buffer (not a parameter, but part of state)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            output: (batch, seq_len, d_model)
        """
        # Add positional encoding
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

### Visualization

In [ ]:
# Create positional encoding
d_model = 128
max_len = 100

pos_encoding = PositionalEncoding(d_model, max_len)

# Get encoding matrix
pe_matrix = pos_encoding.pe[0].numpy()  # (max_len, d_model)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
sns.heatmap(pe_matrix.T, ax=axes[0], cmap='RdBu', center=0)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
axes[0].set_title('Positional Encoding Heatmap')

# Line plot for specific dimensions
for i in [0, 4, 8, 16, 32, 64]:
    axes[1].plot(pe_matrix[:, i], label=f'Dim {i}')

axes[1].set_xlabel('Position')
axes[1].set_ylabel('Encoding Value')
axes[1].set_title('Positional Encoding for Different Dimensions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice:")
print("- Lower dimensions (0, 4, 8): High frequency")
print("- Higher dimensions (32, 64): Low frequency")
print("- This allows model to learn both local and global patterns")

---

## 6. Encoder Layer <a name="encoder"></a>

### Architecture

Each encoder layer consists of:
1. **Multi-Head Self-Attention**
2. **Add & Norm** (residual connection + layer normalization)
3. **Feed-Forward Network**
4. **Add & Norm**

$$
\begin{align}
\text{Attention Output} &= \text{MultiHead}(X, X, X) \\
X' &= \text{LayerNorm}(X + \text{Dropout}(\text{Attention Output})) \\
\text{FFN Output} &= \text{FFN}(X') \\
\text{Output} &= \text{LayerNorm}(X' + \text{Dropout}(\text{FFN Output}))
\end{align}
$$

### Implementation

In [ ]:
class EncoderLayer(nn.Module):
    """
    Single Transformer Encoder Layer.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        
        # Multi-head attention
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        
        # Feed-forward network
        self.ffn = PositionWiseFeedForward(d_model, d_ff, dropout)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: Optional attention mask
        Returns:
            output: (batch, seq_len, d_model)
        """
        # Self-attention with residual and norm
        attn_output, _ = self.self_attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual and norm
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        
        return x

### Complete Encoder (Stack of N layers)

In [ ]:
class TransformerEncoder(nn.Module):
    """
    Complete Transformer Encoder (stack of N encoder layers).
    """
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoder, self).__init__()
        
        # Stack of encoder layers
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: Optional attention mask
        Returns:
            output: (batch, seq_len, d_model)
        """
        for layer in self.layers:
            x = layer(x, mask)
        return x

---

## 7. Decoder Layer <a name="decoder"></a>

### Architecture

Each decoder layer has **three** sub-layers:
1. **Masked Multi-Head Self-Attention**
2. **Multi-Head Cross-Attention** (attend to encoder output)
3. **Feed-Forward Network**

Each with Add & Norm.

### Implementation

In [ ]:
class DecoderLayer(nn.Module):
    """
    Single Transformer Decoder Layer.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        
        # Masked self-attention
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        
        # Cross-attention (attend to encoder output)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        
        # Feed-forward network
        self.ffn = PositionWiseFeedForward(d_model, d_ff, dropout)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        """
        Args:
            x: Decoder input (batch, tgt_seq_len, d_model)
            encoder_output: Encoder output (batch, src_seq_len, d_model)
            src_mask: Mask for encoder output
            tgt_mask: Causal mask for decoder
        Returns:
            output: (batch, tgt_seq_len, d_model)
        """
        # Masked self-attention
        self_attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_output))
        
        # Cross-attention (Q from decoder, K and V from encoder)
        cross_attn_output, _ = self.cross_attention(
            x, encoder_output, encoder_output, src_mask
        )
        x = self.norm2(x + self.dropout(cross_attn_output))
        
        # Feed-forward
        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_output))
        
        return x

### Complete Decoder

In [ ]:
class TransformerDecoder(nn.Module):
    """
    Complete Transformer Decoder (stack of N decoder layers).
    """
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerDecoder, self).__init__()
        
        # Stack of decoder layers
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        """
        Args:
            x: Decoder input (batch, tgt_seq_len, d_model)
            encoder_output: Encoder output (batch, src_seq_len, d_model)
            src_mask: Mask for encoder output
            tgt_mask: Causal mask for decoder
        Returns:
            output: (batch, tgt_seq_len, d_model)
        """
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return x

---

## 8. Complete Transformer <a name="complete"></a>

### Full Architecture

Now we combine everything:
1. Input embedding + positional encoding
2. Encoder stack
3. Decoder stack
4. Output projection

### Implementation

In [ ]:
class Transformer(nn.Module):
    """
    Complete Transformer model.
    """
    def __init__(self, 
                 src_vocab_size,
                 tgt_vocab_size,
                 d_model=512,
                 num_heads=8,
                 num_encoder_layers=6,
                 num_decoder_layers=6,
                 d_ff=2048,
                 dropout=0.1,
                 max_len=5000):
        super(Transformer, self).__init__()
        
        # Embeddings
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Encoder and Decoder
        self.encoder = TransformerEncoder(
            num_encoder_layers, d_model, num_heads, d_ff, dropout
        )
        self.decoder = TransformerDecoder(
            num_decoder_layers, d_model, num_heads, d_ff, dropout
        )
        
        # Output projection
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        
        # Initialize parameters
        self._init_parameters()
        
        self.d_model = d_model
        
    def _init_parameters(self):
        """Xavier initialization."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """
        Args:
            src: Source sequence (batch, src_seq_len)
            tgt: Target sequence (batch, tgt_seq_len)
            src_mask: Source mask
            tgt_mask: Target mask (causal)
        Returns:
            output: (batch, tgt_seq_len, tgt_vocab_size)
        """
        # Embed and add positional encoding
        src_emb = self.src_embedding(src) * np.sqrt(self.d_model)
        tgt_emb = self.tgt_embedding(tgt) * np.sqrt(self.d_model)
        
        src_emb = self.pos_encoding(src_emb)
        tgt_emb = self.pos_encoding(tgt_emb)
        
        # Encoder
        encoder_output = self.encoder(src_emb, src_mask)
        
        # Decoder
        decoder_output = self.decoder(tgt_emb, encoder_output, src_mask, tgt_mask)
        
        # Output projection
        output = self.output_projection(decoder_output)
        
        return output
    
    def encode(self, src, src_mask=None):
        """Encode source sequence."""
        src_emb = self.src_embedding(src) * np.sqrt(self.d_model)
        src_emb = self.pos_encoding(src_emb)
        return self.encoder(src_emb, src_mask)
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        """Decode target sequence."""
        tgt_emb = self.tgt_embedding(tgt) * np.sqrt(self.d_model)
        tgt_emb = self.pos_encoding(tgt_emb)
        decoder_output = self.decoder(tgt_emb, encoder_output, src_mask, tgt_mask)
        return self.output_projection(decoder_output)

### Model Summary

In [ ]:
# Create small Transformer for testing
model = Transformer(
    src_vocab_size=1000,
    tgt_vocab_size=1000,
    d_model=128,
    num_heads=8,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=512,
    dropout=0.1
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass
batch_size = 2
src_len = 10
tgt_len = 8

src = torch.randint(0, 1000, (batch_size, src_len))
tgt = torch.randint(0, 1000, (batch_size, tgt_len))

# Create causal mask for decoder
tgt_mask = create_causal_mask(tgt_len)

output = model(src, tgt, tgt_mask=tgt_mask)

print(f"\nInput shapes:")
print(f"  Source: {src.shape}")
print(f"  Target: {tgt.shape}")
print(f"\nOutput shape: {output.shape}")
print(f"  (batch, tgt_len, vocab_size)")

---

## 9. Training Example: Sequence Copying <a name="training"></a>

### Task

Learn to copy a sequence: `[1, 5, 3, 7] → [1, 5, 3, 7]`

Simple task to verify the implementation works.

### Data Generation

In [ ]:
def generate_copy_data(batch_size, seq_len, vocab_size, num_batches):
    """
    Generate data for sequence copying task.
    """
    data = []
    for _ in range(num_batches):
        # Random sequences
        src = torch.randint(1, vocab_size, (batch_size, seq_len))
        # Target is the same (copy task)
        tgt_input = torch.cat([torch.zeros(batch_size, 1, dtype=torch.long), src[:, :-1]], dim=1)
        tgt_output = src
        
        data.append((src, tgt_input, tgt_output))
    
    return data

### Training Loop

In [ ]:
# Hyperparameters
VOCAB_SIZE = 50
SEQ_LEN = 10
BATCH_SIZE = 32
NUM_BATCHES = 100
EPOCHS = 20

# Create model
model = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=64,
    num_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=256,
    dropout=0.1
)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Generate data
train_data = generate_copy_data(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE, NUM_BATCHES)

# Create causal mask
tgt_mask = create_causal_mask(SEQ_LEN)

# Training
losses = []
accuracies = []

print("Training Transformer on copy task...\n")

for epoch in range(EPOCHS):
    epoch_loss = 0
    epoch_acc = 0
    
    for src, tgt_input, tgt_output in train_data:
        # Forward pass
        output = model(src, tgt_input, tgt_mask=tgt_mask)
        
        # Reshape for loss calculation
        output = output.reshape(-1, VOCAB_SIZE)
        tgt_output = tgt_output.reshape(-1)
        
        # Compute loss
        loss = criterion(output, tgt_output)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Compute accuracy
        predictions = output.argmax(dim=-1)
        acc = (predictions == tgt_output).float().mean()
        
        epoch_loss += loss.item()
        epoch_acc += acc.item()
    
    avg_loss = epoch_loss / NUM_BATCHES
    avg_acc = epoch_acc / NUM_BATCHES
    
    losses.append(avg_loss)
    accuracies.append(avg_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}, Accuracy: {avg_acc:.4f}")

print("\nTraining complete!")

### Visualize Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(losses, linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(accuracies, linewidth=2, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Test Inference

In [ ]:
# Test with new sequences
model.eval()

test_sequences = [
    [5, 12, 7, 23, 8, 15, 3, 19, 11, 6],
    [42, 3, 17, 9, 31, 14, 22, 7, 18, 25],
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
]

print("Testing Transformer on new sequences:\n")

for i, seq in enumerate(test_sequences):
    src = torch.tensor([seq])
    
    # Greedy decoding
    tgt_input = torch.zeros(1, 1, dtype=torch.long)  # Start token
    
    for _ in range(SEQ_LEN):
        # Create causal mask for current length
        tgt_len = tgt_input.size(1)
        mask = create_causal_mask(tgt_len)
        
        # Forward pass
        with torch.no_grad():
            output = model(src, tgt_input, tgt_mask=mask)
        
        # Get next token
        next_token = output[0, -1, :].argmax()
        
        # Append to input
        tgt_input = torch.cat([tgt_input, next_token.unsqueeze(0).unsqueeze(0)], dim=1)
    
    predicted = tgt_input[0, 1:].tolist()  # Remove start token
    
    print(f"Test {i+1}:")
    print(f"  Input:     {seq}")
    print(f"  Predicted: {predicted}")
    print(f"  Match: {seq == predicted}\n")

---

## 10. Attention Visualization <a name="visualization"></a>

### Extract Attention Weights

In [ ]:
# Modified model to return attention weights
class TransformerWithAttention(Transformer):
    """Transformer that returns attention weights."""
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # Store attention weights
        self.encoder_attentions = []
        self.decoder_self_attentions = []
        self.decoder_cross_attentions = []
        
        # Embeddings
        src_emb = self.src_embedding(src) * np.sqrt(self.d_model)
        tgt_emb = self.tgt_embedding(tgt) * np.sqrt(self.d_model)
        
        src_emb = self.pos_encoding(src_emb)
        tgt_emb = self.pos_encoding(tgt_emb)
        
        # Encoder (modified to capture attention)
        x = src_emb
        for layer in self.encoder.layers:
            attn_output, attn_weights = layer.self_attention(x, x, x, src_mask)
            self.encoder_attentions.append(attn_weights)
            x = layer.norm1(x + layer.dropout(attn_output))
            ffn_output = layer.ffn(x)
            x = layer.norm2(x + layer.dropout(ffn_output))
        
        encoder_output = x
        
        # Decoder (modified to capture attention)
        x = tgt_emb
        for layer in self.decoder.layers:
            # Self-attention
            self_attn_output, self_attn_weights = layer.self_attention(x, x, x, tgt_mask)
            self.decoder_self_attentions.append(self_attn_weights)
            x = layer.norm1(x + layer.dropout(self_attn_output))
            
            # Cross-attention
            cross_attn_output, cross_attn_weights = layer.cross_attention(
                x, encoder_output, encoder_output, src_mask
            )
            self.decoder_cross_attentions.append(cross_attn_weights)
            x = layer.norm2(x + layer.dropout(cross_attn_output))
            
            # FFN
            ffn_output = layer.ffn(x)
            x = layer.norm3(x + layer.dropout(ffn_output))
        
        decoder_output = x
        output = self.output_projection(decoder_output)
        
        return output

### Visualize Attention Patterns

In [ ]:
# Create test input
src = torch.tensor([[5, 12, 7, 23, 8, 15, 3, 19]])
tgt = torch.tensor([[0, 5, 12, 7, 23, 8, 15, 3]])

tgt_mask = create_causal_mask(tgt.size(1))

# Create model with attention extraction
model_attn = TransformerWithAttention(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=64,
    num_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=256
)

model_attn.eval()
with torch.no_grad():
    output = model_attn(src, tgt, tgt_mask=tgt_mask)

# Visualize encoder self-attention (layer 0, head 0)
encoder_attn = model_attn.encoder_attentions[0][0, 0].numpy()

plt.figure(figsize=(8, 6))
sns.heatmap(encoder_attn, cmap='YlOrRd', square=True, cbar_kws={'label': 'Attention Weight'})
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Encoder Self-Attention (Layer 1, Head 1)')
plt.show()

# Visualize decoder cross-attention
cross_attn = model_attn.decoder_cross_attentions[0][0, 0].numpy()

plt.figure(figsize=(8, 6))
sns.heatmap(cross_attn, cmap='YlGnBu', square=True, cbar_kws={'label': 'Attention Weight'})
plt.xlabel('Source Position (Encoder)')
plt.ylabel('Target Position (Decoder)')
plt.title('Decoder Cross-Attention (Layer 1, Head 1)')
plt.show()

---

## 11. Complexity Analysis <a name="complexity"></a>

### Time Complexity

For a sequence of length $n$ and model dimension $d$:

| Operation | Complexity | Dominant Factor |
|-----------|-----------|----------------|
| Self-Attention | $O(n^2 \cdot d)$ | Quadratic in sequence length |
| Feed-Forward | $O(n \cdot d^2)$ | Linear in sequence length |
| **Total per layer** | $O(n^2 \cdot d + n \cdot d^2)$ | |

**Bottleneck**: Self-attention becomes expensive for long sequences!

### Space Complexity

- Attention weights: $O(n^2)$ per head
- Parameters: $O(d^2)$ per layer

### Empirical Analysis

In [ ]:
def benchmark_transformer(seq_lengths, d_model=128, num_heads=8):
    """
    Benchmark Transformer for different sequence lengths.
    """
    times = []
    
    model = Transformer(
        src_vocab_size=1000,
        tgt_vocab_size=1000,
        d_model=d_model,
        num_heads=num_heads,
        num_encoder_layers=1,
        num_decoder_layers=1,
        d_ff=d_model*4
    )
    model.eval()
    
    for n in seq_lengths:
        src = torch.randint(0, 1000, (1, n))
        tgt = torch.randint(0, 1000, (1, n))
        tgt_mask = create_causal_mask(n)
        
        # Warm-up
        with torch.no_grad():
            _ = model(src, tgt, tgt_mask=tgt_mask)
        
        # Benchmark
        start = time.time()
        with torch.no_grad():
            for _ in range(10):
                _ = model(src, tgt, tgt_mask=tgt_mask)
        elapsed = (time.time() - start) / 10
        
        times.append(elapsed)
        print(f"Seq length {n:4d}: {elapsed*1000:.2f} ms")
    
    return times

# Benchmark
seq_lengths = [16, 32, 64, 128, 256, 512]
print("Benchmarking Transformer...\n")
times = benchmark_transformer(seq_lengths)

### Visualize Complexity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].plot(seq_lengths, times, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Transformer Inference Time')
axes[0].grid(True, alpha=0.3)

# Log-log scale
axes[1].loglog(seq_lengths, times, 'o-', linewidth=2, markersize=8)

# Fit O(n^2) curve
n_arr = np.array(seq_lengths)
t_arr = np.array(times)
coeffs = np.polyfit(np.log(n_arr), np.log(t_arr), 1)
slope = coeffs[0]

fitted = np.exp(coeffs[1]) * n_arr ** slope
axes[1].loglog(n_arr, fitted, '--', linewidth=2, label=f'Fitted: O(n^{slope:.2f})')

axes[1].set_xlabel('Sequence Length (log scale)')
axes[1].set_ylabel('Time (log scale)')
axes[1].set_title('Complexity Analysis')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nEmpirical complexity: O(n^{slope:.2f})")
print("Expected: O(n^2) for self-attention")

---

## 12. Gradient Checking <a name="gradcheck"></a>

### Verify Backpropagation Correctness

Compare analytical gradients (from autograd) with numerical gradients.

In [ ]:
def gradient_check_transformer():
    """
    Gradient checking for Transformer.
    """
    # Small model for testing
    model = Transformer(
        src_vocab_size=20,
        tgt_vocab_size=20,
        d_model=16,
        num_heads=2,
        num_encoder_layers=1,
        num_decoder_layers=1,
        d_ff=64,
        dropout=0.0  # No dropout for gradient checking
    )
    
    # Small input
    src = torch.randint(0, 20, (1, 4))
    tgt_input = torch.randint(0, 20, (1, 4))
    tgt_output = torch.randint(0, 20, (1, 4))
    
    tgt_mask = create_causal_mask(4)
    
    # Loss function
    criterion = nn.CrossEntropyLoss()
    
    # Forward + backward
    output = model(src, tgt_input, tgt_mask=tgt_mask)
    loss = criterion(output.reshape(-1, 20), tgt_output.reshape(-1))
    loss.backward()
    
    # Check one parameter (e.g., first encoder attention W_q)
    param = model.encoder.layers[0].self_attention.W_q.weight
    analytical_grad = param.grad.clone()
    
    # Numerical gradient (finite differences)
    epsilon = 1e-5
    numerical_grad = torch.zeros_like(param)
    
    # Sample a few random positions
    num_samples = 10
    indices = [(np.random.randint(0, param.shape[0]), 
                np.random.randint(0, param.shape[1])) 
               for _ in range(num_samples)]
    
    errors = []
    
    for i, j in indices:
        # f(θ + ε)
        param.data[i, j] += epsilon
        output_plus = model(src, tgt_input, tgt_mask=tgt_mask)
        loss_plus = criterion(output_plus.reshape(-1, 20), tgt_output.reshape(-1))
        
        # f(θ - ε)
        param.data[i, j] -= 2 * epsilon
        output_minus = model(src, tgt_input, tgt_mask=tgt_mask)
        loss_minus = criterion(output_minus.reshape(-1, 20), tgt_output.reshape(-1))
        
        # Restore
        param.data[i, j] += epsilon
        
        # Numerical gradient
        numerical = (loss_plus.item() - loss_minus.item()) / (2 * epsilon)
        analytical = analytical_grad[i, j].item()
        
        # Relative error
        relative_error = abs(numerical - analytical) / (abs(numerical) + abs(analytical) + 1e-8)
        errors.append(relative_error)
        
        print(f"Position ({i:2d}, {j:2d}): Numerical={numerical:+.6f}, "
              f"Analytical={analytical:+.6f}, RelError={relative_error:.2e}")
    
    avg_error = np.mean(errors)
    print(f"\nAverage relative error: {avg_error:.2e}")
    
    if avg_error < 1e-5:
        print("✅ Gradient check PASSED!")
    else:
        print("⚠️ Gradient check WARNING: Error is high")

# Run gradient check
print("Gradient Checking Transformer...\n")
gradient_check_transformer()

---

## Summary and Key Takeaways

### What We Implemented

1. ✅ **Scaled Dot-Product Attention**: Core mechanism with scaling and masking
2. ✅ **Multi-Head Attention**: Parallel attention heads for diverse relationships
3. ✅ **Position-wise FFN**: Non-linear transformations
4. ✅ **Positional Encoding**: Sinusoidal position information
5. ✅ **Encoder**: Stack of encoder layers with self-attention
6. ✅ **Decoder**: Stack of decoder layers with masked self-attention and cross-attention
7. ✅ **Complete Transformer**: Full seq2seq model
8. ✅ **Training**: Copy task demonstration
9. ✅ **Visualization**: Attention patterns
10. ✅ **Analysis**: Complexity and gradient checking

### Key Mathematical Insights

1. **Attention is dot products + softmax**: $\text{softmax}(QK^T/\sqrt{d_k})V$
2. **Scaling prevents saturation**: $\sqrt{d_k}$ keeps gradients healthy
3. **Positional encoding is deterministic**: Sinusoidal patterns encode position
4. **Residual connections are critical**: Enable deep networks (gradient flow)
5. **Layer normalization stabilizes training**: Normalize across features

### Complexity Trade-offs

- **Time**: $O(n^2 d)$ per layer → Problem for long sequences
- **Space**: $O(n^2)$ for attention weights
- **Solutions**: Sparse attention, linear attention, kernel methods

### Next Steps

1. **Optimizations**: Flash Attention, linear attention variants
2. **Advanced PE**: RoPE, ALiBi (covered in Advanced_Topics_2024.md)
3. **Training Techniques**: Warmup, learning rate schedules, gradient clipping
4. **Applications**: Translation, text generation, vision transformers
5. **Modern Variants**: GPT (decoder-only), BERT (encoder-only), T5 (encoder-decoder)

### References

1. **Vaswani et al. (2017)**: *Attention Is All You Need* - Original paper
2. **Illustrated Transformer**: http://jalammar.github.io/illustrated-transformer/
3. **The Annotated Transformer**: http://nlp.seas.harvard.edu/annotated-transformer/
4. **Advanced Topics**: See [Advanced_Topics_2024.md](../../Advanced_Topics_2024.md)

---

**Congratulations!** You've implemented a complete Transformer from scratch. 🎉

You now understand:
- How attention works mathematically
- Why each component is necessary
- How to train and debug transformers
- Complexity trade-offs and optimizations
